Step  1a : Indexing (Document Ingestion)

In [116]:
# Install required packages if not already installed
# pip install youtube-transcript-api

from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound

In [138]:
video_id = "4nBG848oArI" #Only id not full url
try:
    # Create API instance and fetch transcript
    ytt_api = YouTubeTranscriptApi()
    transcript_list = ytt_api.fetch(video_id, languages=['en'])

    # Join the transcript chunks into a single string
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("Transcript is disabled for this video.")
except NoTranscriptFound:
    print("No transcript found for this video.")

Welcome back aliens. My name is Ain Reddi and now we'll talk about spring AI. Now see everyone wants to build AI application or I can say AI based application. So whatever application you have maybe you are building it from scratch or you have your existing product in which you want to add the AI feature or maybe you want to make it AI enabled and the way you can do that is based on different languages. Till this point a lot of people were building projects B using Python is because it has a lot of libraries and then if you want to interact with all these LLM models Python made sense but then if you think about the world where every product is built using different languages and if you talk about the enterprise market most of the products are built with the help of Java here we we don't want to use a AI enabled feature using Python and Java application which is for the enterprise what if you can connect the LLM models or the AI models with your existing applications with a layer in bet

Step 1b - Indexing(text Splitter)

In [142]:
#Text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.create_documents([transcript])
print(f"Number of chunks: {len(chunks)}")
# chunks = chunks[:70]



Number of chunks: 15


In [143]:
#Step 1c and 1d: Create embeddings and store in vector database
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import OpenAIEmbeddings

from langchain_community.vectorstores import FAISS

embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
vectorstore = FAISS.from_documents(chunks, embedding)



In [ ]:
vectorstore.index_to_docstore_id

{0: 'd63c308d-0593-48de-96d4-20b2d9aed0f6',
 1: 'a367e11b-eb24-4258-acc3-447cd99be64f',
 2: '339c2d1f-7e19-4939-b84c-19d09599a06b',
 3: 'ac1df794-4918-477d-92b5-8c4a18f9feff',
 4: 'cb1679ec-b35b-4ce0-a241-64a145aa27f7',
 5: '08ac6334-236e-4760-95ca-3b19d105c396',
 6: '0f2c7b9a-3443-46d8-8270-b68a686cd6e4',
 7: '3c48bd07-68b7-4554-9aa5-20b91baa57d3',
 8: 'f882c658-86ea-4966-9cab-2cb09dacd92a',
 9: '88f29dcc-9a02-4d5f-b890-57fa7874a502',
 10: 'a3b9eabd-00a6-4fa1-bff0-0f539cb6f00b',
 11: '3db77842-9ed7-4645-aa88-ed87bb344170',
 12: 'a0289f42-51b9-4cc0-9379-23beb3480621',
 13: 'eaf17503-5b7d-4da0-9f53-4af9560ce562',
 14: '8a0982fb-7ba8-4f03-bcf3-7dbd27cd46ff',
 15: '0816759a-36ff-4f55-b19c-6ea42e4d5288',
 16: 'bb311aa1-d1cb-4d57-8e21-d1f59de7f48f',
 17: '264f84fa-a18b-455d-bbff-1463ebd46662',
 18: 'ba39c965-a74c-4564-b7e4-91911be2f1f3',
 19: '7aa375d0-954f-45ad-8cd6-0bec0d4ecec9',
 20: 'e225862a-bee3-4023-b12a-c07c22fd8f08',
 21: 'eb95f284-fb54-4081-962f-e6c56539deb5',
 22: '754f03bb-62d3-

In [122]:
vectorstore.get_by_ids(['fa90854d-17f3-455c-8232-b2eefd6dfc1b'])

[]

## Next steps would be to create a retriever and then a RAG chain to answer questions based on the video transcript. 


In [123]:
#STEP 2 Retreival
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [124]:
retriever.invoke("what is elon talking ?")

[Document(id='24b90a87-ab5f-461f-9353-014366c4e48a', metadata={}, page_content=">> thank you. When I think of social media, Elon, I feel like even data suggests that the current incumbents seem to be losing traction amongst the youngest of audience. >> Yeah. >> Even platforms like Instagram, uh, I mean, they're not exactly like Twitter, but platforms across the board. If one had to rework social media and build something bottom up, what do you think could work for the world of tomorrow? >> Well, I mean, I I don't think that much about um about social media to be frank. I mean, it's I can mostly just want to have have something where there's um a in the case of of X, kind of a global town square, >> uh where where people can say what they want to say uh with words, pictures, video um where there's a secure messaging system. We've recently added the ability to to do audio and video calls. Um, so you're really trying to bring the the the world the world together into um a a collective con

#Step 3 :-> Augumentation Part - Using the retrieved chunks to answer a question using a language model 

In [125]:
from  langchain_core.prompts import PromptTemplate
#Step 3 :-> Augumentation Part - Using the retrieved chunks to answer a question using a language model

prompt = PromptTemplate(
    template = """
    you are a helpful assitant 
    Answer only from the provided transcript context .\
        Ifthe context is insufficient to answer the question, say you don't know.

        {context}
        Question: {question}
        """,
        input_variables=["context", "question"]
)



In [127]:
question = "Is the topic of social media discussed in this video ? If yes then what is said about it ?"
retrieved_docs = retriever.invoke(question)
retrieved_docs

[Document(id='24b90a87-ab5f-461f-9353-014366c4e48a', metadata={}, page_content=">> thank you. When I think of social media, Elon, I feel like even data suggests that the current incumbents seem to be losing traction amongst the youngest of audience. >> Yeah. >> Even platforms like Instagram, uh, I mean, they're not exactly like Twitter, but platforms across the board. If one had to rework social media and build something bottom up, what do you think could work for the world of tomorrow? >> Well, I mean, I I don't think that much about um about social media to be frank. I mean, it's I can mostly just want to have have something where there's um a in the case of of X, kind of a global town square, >> uh where where people can say what they want to say uh with words, pictures, video um where there's a secure messaging system. We've recently added the ability to to do audio and video calls. Um, so you're really trying to bring the the the world the world together into um a a collective con

In [128]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text


">> thank you. When I think of social media, Elon, I feel like even data suggests that the current incumbents seem to be losing traction amongst the youngest of audience. >> Yeah. >> Even platforms like Instagram, uh, I mean, they're not exactly like Twitter, but platforms across the board. If one had to rework social media and build something bottom up, what do you think could work for the world of tomorrow? >> Well, I mean, I I don't think that much about um about social media to be frank. I mean, it's I can mostly just want to have have something where there's um a in the case of of X, kind of a global town square, >> uh where where people can say what they want to say uh with words, pictures, video um where there's a secure messaging system. We've recently added the ability to to do audio and video calls. Um, so you're really trying to bring the the the world the world together into um a a collective consciousness and um that that's I guess different from just saying like what is\n

In [129]:
final_prompt = prompt.invoke({"context": context_text, "question": question})
final_prompt

StringPromptValue(text="\n    you are a helpful assitant \n    Answer only from the provided transcript context .        Ifthe context is insufficient to answer the question, say you don't know.\n\n        >> thank you. When I think of social media, Elon, I feel like even data suggests that the current incumbents seem to be losing traction amongst the youngest of audience. >> Yeah. >> Even platforms like Instagram, uh, I mean, they're not exactly like Twitter, but platforms across the board. If one had to rework social media and build something bottom up, what do you think could work for the world of tomorrow? >> Well, I mean, I I don't think that much about um about social media to be frank. I mean, it's I can mostly just want to have have something where there's um a in the case of of X, kind of a global town square, >> uh where where people can say what they want to say uh with words, pictures, video um where there's a secure messaging system. We've recently added the ability to to 

Step 4: Generation : Using the augmented context to answer the question using a language model.

In [130]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
llm = ChatGroq(model="llama-3.1-8b-instant")
gemini = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash")
answer = gemini.invoke(final_prompt)

print(answer)

content='Yes, the topic of social media is discussed in the video.\n\nHere\'s what is said about it:\n*   The interviewer notes that current social media platforms seem to be losing traction among the youngest audience.\n*   Elon Musk states that he doesn\'t think much about social media, but describes X (formerly Twitter) as aiming to be a "global town square" where people can express themselves with words, pictures, and video.\n*   He mentions X having a secure messaging system and recently adding audio and video calls, with the goal of bringing the world together into a "collective consciousness."\n*   Musk also differentiates X\'s goal from simply creating a "dopamine generating video stream," which he calls "brain rot" and a "drug type of thing," stating his goal is not to optimize for that.\n*   He also discusses the operating principle of X to be balanced and centrist, adhering to any country\'s laws but not putting its "thumb on the scale beyond the laws of a country."\n*   Lat

#Building a Chain to answer questions based on the retrieved context
#The chain will have the following steps:
#1. Retrieve the relevant chunks of text from the vector store based on the question.
#2. Format the retrieved chunks into a context string.
#3. Combine the context string with the question to create a final prompt.
#4. Use a language model to generate an answer based on the final prompt.


In [131]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableSequence, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [132]:
parallel_chain = RunnableParallel({   
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [133]:
parallel_chain.invoke("what has been talked about social media?")

{'context': ">> thank you. When I think of social media, Elon, I feel like even data suggests that the current incumbents seem to be losing traction amongst the youngest of audience. >> Yeah. >> Even platforms like Instagram, uh, I mean, they're not exactly like Twitter, but platforms across the board. If one had to rework social media and build something bottom up, what do you think could work for the world of tomorrow? >> Well, I mean, I I don't think that much about um about social media to be frank. I mean, it's I can mostly just want to have have something where there's um a in the case of of X, kind of a global town square, >> uh where where people can say what they want to say uh with words, pictures, video um where there's a secure messaging system. We've recently added the ability to to do audio and video calls. Um, so you're really trying to bring the the the world the world together into um a a collective consciousness and um that that's I guess different from just saying li

In [136]:
parser = StrOutputParser()
main_chain = parallel_chain | prompt | llm | parser
main_chain.get_graph().print_ascii()

           +---------------------------------+         
           | Parallel<context,question>Input |         
           +---------------------------------+         
                    **               ***               
                 ***                    **             
               **                         ***          
+----------------------+                     **        
| VectorStoreRetriever |                      *        
+----------------------+                      *        
            *                                 *        
            *                                 *        
            *                                 *        
    +-------------+                   +-------------+  
    | format_docs |                   | Passthrough |  
    +-------------+*                  +-------------+  
                    **               **                
                      ***         ***                  
                         **     **              

In [137]:
main_chain.invoke("Can you summarize the video 5 words?")

'AI and robotics future predictions.'